# Raio-X do Parlamentar

Este notebook constroi um **perfil completo** de um parlamentar brasileiro a partir de fontes publicas.

**Fontes utilizadas:**
- TSE (Tribunal Superior Eleitoral) — candidaturas e bens declarados
- Receita Federal — QSA (participacoes societarias)
- CGU / Portal da Transparencia — sancoes (CEIS, CNEP, CEPIM) e gastos parlamentares

**Campos-ponte:** CPF, CNPJ, Nome

**Dificuldade:** Avancado

---

## 1. Setup e imports

Instale as dependencias com `pip install -r requirements.txt` antes de executar.

In [ ]:
import pandas as pd
import requests
import time
from pathlib import Path
from tabulate import tabulate

# ============================================================
# CONFIGURACAO
# ============================================================

# TODO: Insira sua chave de API do Portal da Transparencia
# Obtenha gratuitamente em: https://portaldatransparencia.gov.br/api-de-dados
API_KEY = "SEU_TOKEN_AQUI"

BASE_URL = "https://api.portaldatransparencia.gov.br/api-de-dados"
HEADERS = {"chave-api-dados": API_KEY, "Accept": "application/json"}

# TODO: Informe o nome ou CPF do parlamentar a investigar
NOME_PARLAMENTAR = "NOME COMPLETO DO PARLAMENTAR"
CPF_PARLAMENTAR = ""  # Opcional: preencher se disponivel

# Diretorio com dados baixados da Receita Federal
# Download: https://dados.rfb.gov.br/CNPJ/
DIRETORIO_RFB = Path("./dados_rfb")

# Diretorio com dados do TSE
# Download: https://dadosabertos.tse.jus.br/
DIRETORIO_TSE = Path("./dados_tse")

print("Configuracao carregada.")
print(f"Parlamentar: {NOME_PARLAMENTAR}")

## 2. Buscar candidatura no TSE

Consultar os dados de candidatura do parlamentar no TSE para obter informacoes basicas: partido, cargo, resultado, numero de candidato.

**Fonte:** [TSE — Dados Abertos](https://dadosabertos.tse.jus.br/)  
**Arquivo:** `consulta_cand_YYYY_BRASIL.csv` (substitua YYYY pelo ano da eleicao)

In [ ]:
# TODO: Buscar dados de candidatura no TSE
# Endpoint/arquivo: https://dadosabertos.tse.jus.br/dataset/candidatos
# Arquivo CSV: consulta_cand_2022_BRASIL.csv (ou outro ano eleitoral)

# Exemplo de carregamento:
# df_candidatos = pd.read_csv(
#     DIRETORIO_TSE / "consulta_cand_2022_BRASIL.csv",
#     sep=";",
#     encoding="latin-1",
#     dtype=str,
# )

# TODO: Filtrar pelo nome do parlamentar
# parlamentar = df_candidatos[
#     df_candidatos["NM_CANDIDATO"].str.contains(NOME_PARLAMENTAR, case=False, na=False)
# ]

# TODO: Extrair campos relevantes
# - NR_CPF_CANDIDATO (CPF)
# - NM_CANDIDATO (nome)
# - SG_PARTIDO (partido)
# - DS_CARGO (cargo concorrido)
# - DS_SIT_TOT_TURNO (resultado: eleito, suplente, nao eleito)
# - NR_CANDIDATO (numero de urna)
# - SG_UF (estado)

print("TODO: Carregar e filtrar dados de candidatura do TSE")

## 3. Consultar patrimonio declarado (TSE — Bens de candidato)

O TSE exige que candidatos declarem seus bens ao se registrar. Esses dados permitem acompanhar a **evolucao patrimonial** ao longo das eleicoes.

**Fonte:** [TSE — Prestacao de Contas](https://dadosabertos.tse.jus.br/)  
**Arquivo:** `bem_candidato_YYYY_BRASIL.csv`

In [ ]:
# TODO: Consultar bens declarados pelo parlamentar
# Endpoint/arquivo: https://dadosabertos.tse.jus.br/dataset/prestacao-de-contas-eleitorais
# Arquivo CSV: bem_candidato_2022_BRASIL.csv

# Exemplo de carregamento:
# df_bens = pd.read_csv(
#     DIRETORIO_TSE / "bem_candidato_2022_BRASIL.csv",
#     sep=";",
#     encoding="latin-1",
#     dtype=str,
# )

# TODO: Filtrar bens do parlamentar pelo CPF ou sequencial de candidato
# bens_parlamentar = df_bens[
#     df_bens["SQ_CANDIDATO"] == sequencial_candidato
# ]

# TODO: Calcular patrimonio total
# - DS_TIPO_BEM_CANDIDATO (tipo do bem: imovel, veiculo, aplicacao, etc.)
# - DS_BEM_CANDIDATO (descricao do bem)
# - VR_BEM_CANDIDATO (valor declarado)

# TODO: Para evolucao patrimonial, repetir para eleicoes anteriores (2018, 2014, etc.)

print("TODO: Carregar e analisar bens declarados ao TSE")

## 4. Consultar empresas vinculadas (Receita Federal — QSA)

Verificar se o parlamentar e socio ou administrador de empresas, usando o **Quadro de Socios e Administradores (QSA)** da Receita Federal.

**Fonte:** [Receita Federal — QSA](https://dados.rfb.gov.br/CNPJ/)  
**Arquivos:** `Socios0.csv` a `Socios9.csv`

> **Nota:** O CPF dos socios e parcialmente ocultado na base publica. Use o **nome** como campo de busca principal.

In [ ]:
# TODO: Buscar participacoes societarias no QSA
# Download dos dados: https://dados.rfb.gov.br/CNPJ/dados_abertos_cnpj/
# Arquivos: Socios0.csv ... Socios9.csv (sem cabecalho)

COLUNAS_SOCIOS = [
    "cnpj_basico", "identificador_socio", "nome_socio_razao_social",
    "cpf_cnpj_socio", "qualificacao_socio", "data_entrada_sociedade",
    "pais", "representante_legal", "nome_representante",
    "qualificacao_representante_legal", "faixa_etaria",
]

# TODO: Carregar todos os arquivos de socios
# dfs = []
# for i in range(10):
#     caminho = DIRETORIO_RFB / f"Socios{i}.csv"
#     if caminho.exists():
#         df = pd.read_csv(
#             caminho, sep=";", header=None, names=COLUNAS_SOCIOS,
#             dtype=str, encoding="latin-1",
#         )
#         dfs.append(df)
# df_socios = pd.concat(dfs, ignore_index=True)

# TODO: Buscar pelo nome do parlamentar
# df_socios["nome_norm"] = df_socios["nome_socio_razao_social"].str.upper().str.strip()
# empresas_parlamentar = df_socios[
#     df_socios["nome_norm"].str.contains(NOME_PARLAMENTAR.upper(), na=False)
# ]

# TODO: Enriquecer com dados cadastrais (CNPJ Completa)
# Download: https://dados.rfb.gov.br/CNPJ/dados_abertos_cnpj/
# Arquivos: Empresas0.csv ... Empresas9.csv

# TODO: Exibir lista de empresas encontradas
# print(tabulate(empresas_parlamentar[["cnpj_basico", "nome_socio_razao_social",
#     "qualificacao_socio", "data_entrada_sociedade"]], headers="keys"))

print("TODO: Carregar QSA e buscar empresas do parlamentar")

## 5. Verificar sancoes (CEIS / CNEP / CEPIM)

Verificar se o parlamentar ou suas empresas constam nos cadastros de sancoes da CGU:

- **CEIS** — Cadastro de Empresas Inidoneas e Suspensas
- **CNEP** — Cadastro Nacional de Empresas Punidas (Lei Anticorrupcao)
- **CEPIM** — Cadastro de Entidades Privadas Sem Fins Lucrativos Impedidas

**API:** `https://api.portaldatransparencia.gov.br/api-de-dados`

In [ ]:
# TODO: Verificar sancoes no CEIS, CNEP e CEPIM
# Endpoints:
#   CEIS: GET https://api.portaldatransparencia.gov.br/api-de-dados/ceis
#         Params: cnpjSancionado={CNPJ}
#   CNEP: GET https://api.portaldatransparencia.gov.br/api-de-dados/cnep
#         Params: cnpjSancionado={CNPJ}
#   CEPIM: GET https://api.portaldatransparencia.gov.br/api-de-dados/cepim
#          Params: cnpjSancionado={CNPJ}

def verificar_sancao(endpoint: str, cnpj: str) -> list:
    """Consulta um cadastro de sancoes por CNPJ."""
    # TODO: Implementar consulta
    # try:
    #     resp = requests.get(
    #         f"{BASE_URL}/{endpoint}",
    #         headers=HEADERS,
    #         params={"cnpjSancionado": cnpj, "pagina": 1},
    #         timeout=30,
    #     )
    #     resp.raise_for_status()
    #     return resp.json()
    # except Exception as e:
    #     print(f"Erro ao consultar {endpoint}: {e}")
    #     return []
    pass

# TODO: Para cada empresa do parlamentar (passo anterior), verificar:
# - CEIS: verificar_sancao("ceis", cnpj)
# - CNEP: verificar_sancao("cnep", cnpj)
# - CEPIM: verificar_sancao("cepim", cnpj)

# TODO: Consolidar resultados em tabela de sancoes
# sancoes = []
# for cnpj in cnpjs_empresas_parlamentar:
#     for cadastro in ["ceis", "cnep", "cepim"]:
#         resultado = verificar_sancao(cadastro, cnpj)
#         if resultado:
#             sancoes.extend(resultado)
#     time.sleep(2)  # Respeitar rate limit (30 req/min)

print("TODO: Verificar sancoes das empresas do parlamentar")

## 6. Consultar gastos parlamentares (Portal da Transparencia)

Buscar gastos do parlamentar, incluindo:
- **Emendas parlamentares** — recursos direcionados pelo parlamentar
- **Cartao de pagamento** — gastos com cartao corporativo (quando aplicavel)

**API:** `https://api.portaldatransparencia.gov.br/api-de-dados`

In [ ]:
# TODO: Consultar emendas parlamentares
# Endpoint: GET https://api.portaldatransparencia.gov.br/api-de-dados/emendas
# Params: nomeAutor={NOME_PARLAMENTAR}, ano={ANO}

def buscar_emendas(nome_autor: str, ano: int) -> list:
    """Busca emendas parlamentares por nome do autor."""
    # TODO: Implementar busca paginada
    # todos = []
    # for pagina in range(1, 20):
    #     resp = requests.get(
    #         f"{BASE_URL}/emendas",
    #         headers=HEADERS,
    #         params={
    #             "nomeAutor": nome_autor,
    #             "ano": ano,
    #             "pagina": pagina,
    #         },
    #         timeout=30,
    #     )
    #     resp.raise_for_status()
    #     dados = resp.json()
    #     if not dados:
    #         break
    #     todos.extend(dados)
    #     time.sleep(2)
    # return todos
    pass

# TODO: Buscar emendas dos ultimos anos
# emendas = []
# for ano in range(2023, 2026):
#     emendas.extend(buscar_emendas(NOME_PARLAMENTAR, ano))

# TODO: Analisar:
# - Valor total de emendas
# - Destino das emendas (municipio, orgao)
# - Se algum destino coincide com empresa do parlamentar (passo 4)

print("TODO: Consultar e analisar gastos parlamentares")

## 7. Consolidar perfil completo

Reunir todas as informacoes coletadas em um perfil unico do parlamentar, com indicadores de risco e resumo executivo.

In [ ]:
# TODO: Consolidar todas as informacoes em um perfil

# perfil = {
#     "nome": NOME_PARLAMENTAR,
#     "cpf": CPF_PARLAMENTAR,
#     "partido": "",           # Do passo 2 (TSE)
#     "cargo": "",             # Do passo 2 (TSE)
#     "uf": "",                # Do passo 2 (TSE)
#     "patrimonio_declarado": 0.0,  # Do passo 3 (TSE bens)
#     "num_empresas": 0,       # Do passo 4 (QSA)
#     "empresas": [],          # Do passo 4 (QSA + CNPJ)
#     "sancoes": [],           # Do passo 5 (CEIS/CNEP/CEPIM)
#     "total_emendas": 0.0,    # Do passo 6 (Emendas)
#     "emendas": [],           # Do passo 6 (Emendas)
# }

# TODO: Calcular indicadores de risco
# riscos = []
# if perfil["num_empresas"] > 0:
#     riscos.append("Parlamentar possui participacoes societarias")
# if perfil["sancoes"]:
#     riscos.append(f"Empresas com {len(perfil['sancoes'])} sancao(oes) ativa(s)")
# # Verificar se emendas beneficiam localidade das empresas
# # Verificar evolucao patrimonial anomala

# TODO: Exibir perfil consolidado
# print("=" * 60)
# print(f"RAIO-X: {perfil['nome']}")
# print(f"Partido: {perfil['partido']} | Cargo: {perfil['cargo']} | UF: {perfil['uf']}")
# print(f"Patrimonio declarado: R$ {perfil['patrimonio_declarado']:,.2f}")
# print(f"Empresas vinculadas: {perfil['num_empresas']}")
# print(f"Sancoes encontradas: {len(perfil['sancoes'])}")
# print(f"Total em emendas: R$ {perfil['total_emendas']:,.2f}")
# print("=" * 60)
# if riscos:
#     print("\nINDICADORES DE RISCO:")
#     for r in riscos:
#         print(f"  [!] {r}")

print("TODO: Consolidar perfil e gerar relatorio")